# 🧠 Psychology NLP Chatbot
### Based on NCERT Class XI Psychology Textbook (Chapter 1)
This chatbot answers questions about psychology using a retrieval-based approach with TF-IDF similarity — a much more reliable method than LSTM classification for this type of knowledge-based Q&A.

## Step 1 — Install & Import Libraries

In [ ]:
# Install required libraries (run once)
# !pip install pdfplumber nltk scikit-learn --quiet

In [1]:
# =========================
# IMPORT LIBRARIES
# =========================

import pdfplumber
import nltk
import numpy as np
import re
import string

from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
print("✅ All libraries imported successfully.")

✅ All libraries imported successfully.


## Step 2 — Load the PDF
Update `pdf_path` to the location of your psychology PDF file.  
- **Google Colab**: upload the file first, then use `pdf_path = "physicology_1st.pdf"`  
- **Local machine**: use the full path, e.g. `r"C:\Users\YourName\Downloads\physicology_1st.pdf"`

In [2]:
# =========================
# LOAD PDF
# =========================

# ✏️  UPDATE THIS PATH to wherever your PDF is saved
pdf_path = r"C:\Users\Sahil\Downloads\physicology_!st.pdf"   # <-- change this

text = ""

try:
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + " "

    print(f"✅ PDF loaded successfully.")
    print(f"   Total characters extracted: {len(text):,}")
    print(f"\nFirst 300 characters preview:\n{text[:300]}")

except FileNotFoundError:
    print("❌ ERROR: PDF file not found.")
    print(f"   Looked for: {pdf_path}")
    print("   Please update pdf_path to the correct location of your file.")

✅ PDF loaded successfully.
   Total characters extracted: 61,813

First 300 characters preview:
1
WWhhaatt iiss PPssyycchhoollooggyy??
Chapter
After reading this chapter, you would be able to
• understand the nature and role of psychology in understanding mind
and behaviour,
• state the growth of the discipline,
• know the different fields of psychology, its relationship with other
disciplines


## Step 3 — Clean and Tokenize Text

In [3]:
# =========================
# TEXT CLEANING FUNCTION
# =========================

def clean_text(text):
    """Lowercase, remove special characters and extra whitespace."""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)              # collapse whitespace
    text = re.sub(r'[^a-z0-9\s.,?!\'\-]', '', text)  # keep basic punctuation
    return text.strip()


# =========================
# SENTENCE TOKENIZATION
# =========================

raw_sentences = sent_tokenize(text)

# Keep sentences with more than 8 words (filter out headers, page numbers, etc.)
clean_sentences = []
for sentence in raw_sentences:
    sentence = sentence.strip()
    if len(sentence.split()) > 8:
        clean_sentences.append(sentence)

print(f"✅ Total usable sentences: {len(clean_sentences)}")
print("\nSample sentences:")
for s in clean_sentences[:5]:
    print(f"  → {s[:120]}")

✅ Total usable sentences: 394

Sample sentences:
  → Chapter
After reading this chapter, you would be able to
• understand the nature and role of psychology in understanding
  → Psychology as a Discipline
Psychology as a Natural Science
Psychology as a Social Science
Understanding Mind and Behavio
  → – Norman Cousins
Reprint 2026-27 Introduction
You were, perhaps, asked by your teacher in the first class why you opted 
  → If you were asked this
question, what was your response?
  → Generally, the range of responses which surface
in class to this question are truly bewildering.


## Step 4 — Build Meaningful Q&A Pairs
Instead of splitting sentences in half (which creates non-questions), we generate real questions using keyword patterns found in the text — a far more reliable approach for a psychology textbook.

In [4]:
# =========================
# BUILD MEANINGFUL QA PAIRS
# =========================

qa_pairs = []

# Pattern-based question generation from sentences
for sentence in clean_sentences:
    s_lower = sentence.lower()

    # "X is defined as..." → "What is X?"
    if " is defined as" in s_lower or " is defined " in s_lower:
        qa_pairs.append({
            "question": "What is " + sentence.split(" is")[0].strip() + "?",
            "answer": sentence
        })

    # "X was founded/established..." → "When/Who founded X?"
    if "was established" in s_lower or "was founded" in s_lower:
        qa_pairs.append({
            "question": "When was " + sentence.split("was")[0].strip() + " established?",
            "answer": sentence
        })

    # Sentences mentioning key psychology terms → add as direct retrieval candidates
    key_terms = [
        "psychology", "behaviour", "mind", "consciousness", "cognitive",
        "behaviourism", "gestalt", "psychoanalysis", "humanistic",
        "structuralism", "functionalism", "introspection", "wundt",
        "freud", "watson", "skinner", "india", "branches"
    ]
    for term in key_terms:
        if term in s_lower:
            qa_pairs.append({
                "question": sentence,   # store raw sentence for TF-IDF matching
                "answer": sentence
            })
            break  # one entry per sentence is enough

# Remove duplicates
seen = set()
unique_qa = []
for pair in qa_pairs:
    if pair["answer"] not in seen:
        seen.add(pair["answer"])
        unique_qa.append(pair)

questions = [p["question"] for p in unique_qa]
answers   = [p["answer"]   for p in unique_qa]

print(f"✅ Total Q&A pairs created: {len(unique_qa)}")
print("\nSample pairs:")
for p in unique_qa[:4]:
    print(f"  Q: {p['question'][:90]}")
    print(f"  A: {p['answer'][:120]}")
    print()

✅ Total Q&A pairs created: 208

Sample pairs:
  Q: Chapter
After reading this chapter, you would be able to
• understand the nature and role 
  A: Chapter
After reading this chapter, you would be able to
• understand the nature and role of psychology in understanding

  Q: Psychology as a Discipline
Psychology as a Natural Science
Psychology as a Social Science

  A: Psychology as a Discipline
Psychology as a Natural Science
Psychology as a Social Science
Understanding Mind and Behavio

  Q: – Norman Cousins
Reprint 2026-27 Introduction
You were, perhaps, asked by your teacher in 
  A: – Norman Cousins
Reprint 2026-27 Introduction
You were, perhaps, asked by your teacher in the first class why you opted 

  Q: The Indian philosophical traditions, in particular, deal with questions
relating to why pe
  A: The Indian philosophical traditions, in particular, deal with questions
relating to why people behave in the manner in w



## Step 5 — Build TF-IDF Retrieval Index
Instead of training an LSTM classifier (which fails on unique answers), we use **TF-IDF + Cosine Similarity** — the standard and reliable approach for retrieval-based chatbots on small-to-medium corpora.

In [5]:
# =========================
# BUILD TF-IDF INDEX
# =========================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Fit vectorizer on all answer sentences (the knowledge base)
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),       # unigrams + bigrams
    stop_words='english',     # remove common words
    max_features=5000
)

# Use the full sentence pool (clean_sentences) as the retrieval corpus
corpus = clean_sentences
tfidf_matrix = vectorizer.fit_transform(corpus)

print(f"✅ TF-IDF index built.")
print(f"   Corpus size  : {len(corpus)} sentences")
print(f"   Feature count: {tfidf_matrix.shape[1]} n-grams")

✅ TF-IDF index built.
   Corpus size  : 394 sentences
   Feature count: 5000 n-grams


## Step 6 — Chatbot Response Function

In [6]:
# =========================
# CHATBOT FUNCTION
# =========================

def chatbot_response(user_input, top_k=2, threshold=0.10):
    """
    Returns the most relevant answer(s) from the psychology textbook.
    
    Parameters
    ----------
    user_input : str   — the user's question
    top_k      : int   — number of best-matching sentences to return
    threshold  : float — minimum cosine similarity to give a response
    """
    # Clean and vectorize the input
    cleaned_input = clean_text(user_input)
    input_vec = vectorizer.transform([cleaned_input])

    # Compute cosine similarity against the full corpus
    similarities = cosine_similarity(input_vec, tfidf_matrix).flatten()

    # Get top-k indices sorted by similarity (descending)
    top_indices = np.argsort(similarities)[::-1][:top_k]
    best_score  = similarities[top_indices[0]]

    # Low confidence → fallback
    if best_score < threshold:
        return "I'm not sure about that based on the textbook. Could you rephrase your question?"

    # Combine top-k sentences for a richer answer
    response_sentences = []
    seen_texts = set()
    for idx in top_indices:
        if similarities[idx] > threshold:
            sentence = corpus[idx].strip()
            if sentence not in seen_texts:
                response_sentences.append(sentence)
                seen_texts.add(sentence)

    return " ".join(response_sentences)


# Quick test
test_q = "What is psychology?"
print(f"Q: {test_q}")
print(f"A: {chatbot_response(test_q)}\n")

test_q2 = "Who founded behaviourism?"
print(f"Q: {test_q2}")
print(f"A: {chatbot_response(test_q2)}")

Q: What is psychology?
A: Some of the major fields of psychology are: cognitive psychology, biological psychology,
health psychology, developmental psychology, social psychology, educational and school
psychology, clinical and counselling psychology, environmental psychology, industrial/
organisational psychology, sports psychology. psychology, psychology of women, and political
psychology, to name a few.

Q: Who founded behaviourism?
A: He founded psychoanalysis as a as determined by environmental conditions
system to understand and cure psychological undermines human freedom and dignity and
disorders. 1981 David Hubel and Torsten Wiesel win the Nobel
1924 Indian Psychological Association is Prize for their research on vision cells in the
founded.


## Step 7 — Interactive Chat Loop

In [7]:
# =========================
# INTERACTIVE CHAT LOOP
# =========================
# Works in both terminal and Jupyter.
# In Jupyter: type your question in the input box and press Enter.
# Type 'exit' or 'quit' to stop.

print("\n===== 🧠 Psychology Chatbot =====")
print("Ask me anything about psychology (from the NCERT textbook).")
print("Type 'exit' to stop.\n")

while True:
    try:
        user_input = input("You: ").strip()
    except EOFError:
        # Handles non-interactive environments
        print("(Non-interactive environment detected — skipping chat loop)")
        break

    if not user_input:
        continue

    if user_input.lower() in ("exit", "quit"):
        print("Chatbot: Goodbye! Keep learning psychology! 📚")
        break

    response = chatbot_response(user_input)
    print(f"Chatbot: {response}\n")


===== 🧠 Psychology Chatbot =====
Ask me anything about psychology (from the NCERT textbook).
Type 'exit' to stop.



You:  what is pyschology


Chatbot: I'm not sure about that based on the textbook. Could you rephrase your question?



You:  what is pyschology?


Chatbot: I'm not sure about that based on the textbook. Could you rephrase your question?



...

Chatbot: Some of the major fields of psychology are: cognitive psychology, biological psychology,
health psychology, developmental psychology, social psychology, educational and school
psychology, clinical and counselling psychology, environmental psychology, industrial/
organisational psychology, sports psychology. psychology, psychology of women, and political
psychology, to name a few.



You:  Norman Cousins


Chatbot: – Norman Cousins
Reprint 2026-27 Introduction
You were, perhaps, asked by your teacher in the first class why you opted for
psychology over other subjects.



You:  exit


Chatbot: Goodbye! Keep learning psychology! 📚


## Step 8 — Save the Model Artifacts (Optional)

In [8]:
# =========================
# SAVE MODEL ARTIFACTS
# =========================
# Saves the vectorizer and corpus so you can reload without re-running everything.

import pickle

with open("psychology_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("psychology_corpus.pkl", "wb") as f:
    pickle.dump(corpus, f)

print("✅ Vectorizer and corpus saved.")
print("   Reload with:")
print("   vectorizer = pickle.load(open('psychology_vectorizer.pkl','rb'))")
print("   corpus     = pickle.load(open('psychology_corpus.pkl','rb'))")

✅ Vectorizer and corpus saved.
   Reload with:
   vectorizer = pickle.load(open('psychology_vectorizer.pkl','rb'))
   corpus     = pickle.load(open('psychology_corpus.pkl','rb'))


## (Optional) Reload Saved Artifacts

In [ ]:
# =========================
# RELOAD SAVED ARTIFACTS
# =========================

# import pickle
# from sklearn.metrics.pairwise import cosine_similarity
# import numpy as np

# vectorizer   = pickle.load(open("psychology_vectorizer.pkl", "rb"))
# corpus       = pickle.load(open("psychology_corpus.pkl",     "rb"))
# tfidf_matrix = vectorizer.transform(corpus)

# print("✅ Model reloaded — ready to chat!")